## Week 6 Tasks: Feature Engineering & Segment Analysis

### 1. Import Libraries

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# geopandas for adding school district
import geopandas as gpd
from urllib.request import urlretrieve

### 2. Define File Paths and Load Cleaned Datasets

In [2]:
DATA_DIR = Path("cleaned_data")
FEATURE_OUTPUT_DIR = DATA_DIR / "feature_engineering"

FEATURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOLD_FILE = DATA_DIR / "combined_sold_residential_cleaned.csv"
LIST_FILE = DATA_DIR / "combined_list_residential_cleaned.csv"

sold_clean = pd.read_csv(
    SOLD_FILE,
    low_memory=False
)

list_clean = pd.read_csv(
    LIST_FILE,
    low_memory=False
)

print("Sold shape:", sold_clean.shape)
print("List shape:", list_clean.shape)

Sold shape: (447769, 87)
List shape: (615316, 77)


### 3. Check Required Columns

In [3]:
required_sold_columns = [
    "ClosePrice",
    "ListPrice",
    "OriginalListPrice",
    "LivingArea",
    "DaysOnMarket",
    "CloseDate",
    "ListingContractDate",
    "PurchaseContractDate",
    "Latitude",
    "Longitude",
    "PropertyType",
    "PropertySubType",
    "CountyOrParish",
    "MLSAreaMajor",
    "ListOfficeName",
    "BuyerOfficeName"
]

required_list_columns = [
    "ListPrice",
    "OriginalListPrice",
    "LivingArea",
    "DaysOnMarket",
    "ListingContractDate",
    "Latitude",
    "Longitude",
    "PropertyType",
    "PropertySubType",
    "CountyOrParish",
    "MLSAreaMajor",
    "ListOfficeName"
]

missing_sold_columns = [
    col
    for col in required_sold_columns
    if col not in sold_clean.columns
]

missing_list_columns = [
    col
    for col in required_list_columns
    if col not in list_clean.columns
]

print("Missing Sold columns:", missing_sold_columns)
print("Missing List columns:", missing_list_columns)

if missing_sold_columns:
    raise KeyError(
        f"Sold dataset is missing required columns: "
        f"{missing_sold_columns}"
    )

if missing_list_columns:
    raise KeyError(
        f"List dataset is missing required columns: "
        f"{missing_list_columns}"
    )

Missing Sold columns: []
Missing List columns: []


### 4. Prepare Data Types

In [4]:
# copy of the sold/list cleaned data
sold_features = sold_clean.copy()
list_features = list_clean.copy()

#### 4.1 Check datetime columns

In [5]:
datetime_columns = [
    "CloseDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "ContractStatusChangeDate"
]

print("Sold datetime column data types:")
print(
    sold_features[datetime_columns].dtypes
)

print("\nList datetime column data types:")
print(
    list_features[datetime_columns].dtypes
)

Sold datetime column data types:
CloseDate                   object
PurchaseContractDate        object
ListingContractDate         object
ContractStatusChangeDate    object
dtype: object

List datetime column data types:
CloseDate                   object
PurchaseContractDate        object
ListingContractDate         object
ContractStatusChangeDate    object
dtype: object


Because the cleaned data was saved as a CSV file, pandas reads the datetime columns as objects when the file is loaded. Therefore, I need to convert these columns back to the datetime data type before performing feature engineering.

#### 4.2 Convert Date Columns to Datetime

In [6]:
for col in datetime_columns:
    sold_features[col] = pd.to_datetime(
        sold_features[col],
        errors="coerce"
    )

for col in datetime_columns:
    list_features[col] = pd.to_datetime(
        list_features[col],
        errors="coerce"
    )

# check each column's data type if it convert to datetime
print("Sold datetime column data types:")
print(sold_features[datetime_columns].dtypes)

print("\nList datetime column data types:")
print(list_features[datetime_columns].dtypes)

Sold datetime column data types:
CloseDate                   datetime64[ns]
PurchaseContractDate        datetime64[ns]
ListingContractDate         datetime64[ns]
ContractStatusChangeDate    datetime64[ns]
dtype: object

List datetime column data types:
CloseDate                   datetime64[ns]
PurchaseContractDate        datetime64[ns]
ListingContractDate         datetime64[ns]
ContractStatusChangeDate    datetime64[ns]
dtype: object


### 5. Create Market Metrics

In [7]:
sold_features = sold_features.rename(
    columns={"year_month": "source_year_month"}
)

list_features = list_features.rename(
    columns={"year_month": "source_year_month"}
)

The existing `year_month` column was derived from the source filename. It is renamed to `source_year_month` to distinguish it from `YrMo`, which will be derived from the actual closing or listing date for time-series analysis.

#### 5.1 Create Sold Market Metrics

In [8]:
# Sold Price Ratio
sold_features["price_ratio"] = np.where(
    sold_features["ListPrice"].gt(0),
    sold_features["ClosePrice"]
    / sold_features["ListPrice"],
    np.nan
)

# Sold Close to Original List Ratio
sold_features["close_to_original_list_ratio"] = np.where(
    sold_features["OriginalListPrice"].gt(0),
    sold_features["ClosePrice"]
    / sold_features["OriginalListPrice"],
    np.nan
)

# Sold Price per Square Foot
sold_features["price_per_sqft"] = np.where(
    sold_features["LivingArea"].gt(0),
    sold_features["ClosePrice"]
    / sold_features["LivingArea"],
    np.nan
)

# Sold Days on Market
sold_features["days_on_market"] = (
    sold_features["DaysOnMarket"]
)

# Sold Year
sold_features["year"] = (
    sold_features["CloseDate"].dt.year
)

# Sold Month
sold_features["month"] = (
    sold_features["CloseDate"].dt.month
)

# Sold Year_Month
sold_features["yrmo"] = (
    sold_features["CloseDate"].dt.strftime("%Y-%m")
)

# Sold Listing to Contract Days
sold_features["listing_to_contract_days"] = (
    sold_features["PurchaseContractDate"]
    - sold_features["ListingContractDate"]
).dt.days

# Sold Contract to Close Days
sold_features["contract_to_close_days"] = (
    sold_features["CloseDate"]
    - sold_features["PurchaseContractDate"]
).dt.days

#### 5.2 Create List Market Metrics


The Sold and List datasets serve different analytical purposes, so they do not require the same engineered metrics. The Sold dataset contains completed transaction information and is used to calculate closing-price ratios, price per square foot, and transaction duration metrics.

The List dataset is primarily used to analyze new listings. Therefore, its `Year`, `Month`, and `YrMo` fields are derived from `ListingContractDate` rather than `CloseDate`. These fields allow Tableau to group and count new listings by their actual listing month.

Transaction-related metrics, such as `price_ratio`, `close_to_original_list_ratio`, `listing_to_contract_days`, and `contract_to_close_days`, are not created for the List dataset because they depend on completed sale information such as `ClosePrice` and `CloseDate`.

An optional `list_price_per_sqft` feature may be created using `ListPrice / LivingArea` to compare asking prices across properties of different sizes.

In [9]:
# Listing Days on Market
list_features["days_on_market"] = (
    list_features["DaysOnMarket"]
)

# Listing Year
list_features["year"] = (
    list_features["ListingContractDate"].dt.year
)

# Listing Month
list_features["month"] = (
    list_features["ListingContractDate"].dt.month
)

# Listing YrMo
list_features["yrmo"] = (
    list_features["ListingContractDate"].dt.strftime("%Y-%m")
)

# Listing Price Per Square Feet
list_features["list_price_per_sqft"] = np.where(
    list_features["LivingArea"].gt(0),
    list_features["ListPrice"]
    / list_features["LivingArea"],
    np.nan
)

### 6. Validate Engineered Features

Validate the newly created features to confirm that the calculations were completed correctly and that the output is ready for analysis.

#### 6.1 Check Engineered Feature Columns

Confirm that all required engineered columns were successfully created in both the Sold and List datasets.

In [10]:
# create engineered columns for sold & list dataset
sold_engineered_columns = [
    "price_ratio",
    "close_to_original_list_ratio",
    "price_per_sqft",
    "days_on_market",
    "year",
    "month",
    "yrmo",
    "listing_to_contract_days",
    "contract_to_close_days"
]

list_engineered_columns = [
    "days_on_market",
    "year",
    "month",
    "yrmo",
    "list_price_per_sqft"
]

In [11]:
# Make sure columns are existed in Sold and List dataset
missing_sold_features = [
    col
    for col in sold_engineered_columns
    if col not in sold_features.columns
]

missing_list_features = [
    col
    for col in list_engineered_columns
    if col not in list_features.columns
]

print("Missing Sold engineered features:", missing_sold_features)
print("Missing List engineered features:", missing_list_features)

Missing Sold engineered features: []
Missing List engineered features: []


#### 6.2 Review Feature Data Types

Check the data types of the engineered features to ensure that ratios, prices, and duration fields are numeric, while yrmo is stored in an appropriate format for time-series analysis.

In [12]:
print("Sold engineered feature data types:")
print(
    sold_features[sold_engineered_columns].dtypes
)

print("\nList engineered feature data types:")
print(
    list_features[list_engineered_columns].dtypes
)

Sold engineered feature data types:
price_ratio                     float64
close_to_original_list_ratio    float64
price_per_sqft                  float64
days_on_market                    int64
year                              int32
month                             int32
yrmo                             object
listing_to_contract_days        float64
contract_to_close_days          float64
dtype: object

List engineered feature data types:
days_on_market           int64
year                     int32
month                    int32
yrmo                    object
list_price_per_sqft    float64
dtype: object


#### 6.3 Convert `yrmo` to Datetime

The `yrmo` field was initially created using `strftime("%Y-%m")`, which returns text values and causes the column to be stored as an `object` data type.

To support accurate date sorting, filtering, and monthly time-series analysis in Tableau, `yrmo` is instead converted to a monthly datetime value. Each month is represented by the first day of that month, such as `2025-06-01`, while still representing the entire month of June 2025.

For the Sold dataset, `yrmo` is derived from `CloseDate`. For the List dataset, it is derived from `ListingContractDate`.


In [13]:
# Sold dataset

sold_features["yrmo"] = (
    sold_features["CloseDate"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

# List dataset

list_features["yrmo"] = (
    list_features["ListingContractDate"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

In [14]:
print("Sold YrMo dtype:", sold_features["yrmo"].dtype)
print("List YrMo dtype:", list_features["yrmo"].dtype)

Sold YrMo dtype: datetime64[ns]
List YrMo dtype: datetime64[ns]


#### 6.4 Check Missing Values

Review the missing-value counts and percentages for each engineered feature. 

Missing values may occur when the original price, living area, or date fields are missing or invalid.

In [15]:
sold_feature_missing = pd.DataFrame({
    "missing_count":
        sold_features[sold_engineered_columns].isna().sum(),

    "missing_rate_pct":
        sold_features[sold_engineered_columns]
        .isna()
        .mean()
        .mul(100)
        .round(2)
})

display(sold_feature_missing)

,missing_count,missing_rate_pct
price_ratio,2,0.00
close_to_original_list_ratio,826,0.18
price_per_sqft,255,0.06
days_on_market,0,0.00
year,0,0.00
month,0,0.00
yrmo,0,0.00
listing_to_contract_days,199,0.04
contract_to_close_days,198,0.04


In [16]:
list_feature_missing = pd.DataFrame({
    "missing_count":
        list_features[list_engineered_columns].isna().sum(),

    "missing_rate_pct":
        list_features[list_engineered_columns]
        .isna()
        .mean()
        .mul(100)
        .round(2)
})

display(list_feature_missing)

,missing_count,missing_rate_pct
days_on_market,0,0.0
year,0,0.0
month,0,0.0
yrmo,0,0.0
list_price_per_sqft,623,0.1


#### 6.5 Review Summary Statistics

Generate descriptive statistics for the numeric features, including the count, mean, minimum, maximum, and quartiles, to identify unusual or extreme values.

In [17]:
sold_numeric_features = [
    "price_ratio",
    "close_to_original_list_ratio",
    "price_per_sqft",
    "days_on_market",
    "listing_to_contract_days",
    "contract_to_close_days"
]

display(
    sold_features[
        sold_numeric_features
    ].describe().T
)

,count,mean,std,min,25%,50%,75%,max
price_ratio,447767.0,1.080796,8.584952,9.591326e-07,0.976423,1.000000,1.019512,1.153846e+03
close_to_original_list_ratio,446943.0,50.949208,14768.453807,9.207366e-07,0.953753,0.995662,1.019355,9.118000e+06
price_per_sqft,447514.0,646.239980,5047.575729,4.981497e-04,368.018848,537.310001,732.588042,1.164067e+06
days_on_market,447769.0,37.313041,53.603946,0.000000e+00,8.000000,18.000000,48.000000,1.243000e+04
listing_to_contract_days,447570.0,45.149918,86.220132,-3.640700e+04,10.000000,25.000000,58.000000,1.465700e+04
contract_to_close_days,447571.0,31.607421,60.408118,-3.310000e+02,21.000000,29.000000,36.000000,3.662900e+04


#### 6.6 Check Negative Duration Values

Identify negative values in listing_to_contract_days and contract_to_close_days. 

Negative values may indicate inconsistent date sequences and should be flagged for review rather than automatically removed.

In [18]:
print(
    "Negative listing-to-contract days:",
    sold_features[
        "listing_to_contract_days"
    ].lt(0).sum()
)

print(
    "Negative contract-to-close days:",
    sold_features[
        "contract_to_close_days"
    ].lt(0).sum()
)

Negative listing-to-contract days: 289
Negative contract-to-close days: 241


In [19]:
sold_features["negative_listing_to_contract_flag"] = (
    sold_features["listing_to_contract_days"].notna()
    & sold_features["listing_to_contract_days"].lt(0)
)

sold_features["negative_contract_to_close_flag"] = (
    sold_features["contract_to_close_days"].notna()
    & sold_features["contract_to_close_days"].lt(0)
)

#### 6.7 Display Sample Output

Display a sample table containing the original source columns and newly engineered features to confirm that the calculations were populated correctly.

In [89]:
sold_sample_columns = [
    "ListingKey",
    "ClosePrice",
    "ListPrice",
    "OriginalListPrice",
    "LivingArea",
    "price_ratio",
    "close_to_original_list_ratio",
    "price_per_sqft",
    "days_on_market",
    "CloseDate",
    "year",
    "month",
    "yrmo",
    "ListingContractDate",
    "PurchaseContractDate",
    "listing_to_contract_days",
    "contract_to_close_days"
]

existing_sold_sample_columns = [
    col
    for col in sold_sample_columns
    if col in sold_features.columns
]

display(
    sold_features[
        existing_sold_sample_columns
    ].head(10)
)

,ListingKey,ClosePrice,ListPrice,OriginalListPrice,LivingArea,price_ratio,close_to_original_list_ratio,price_per_sqft,days_on_market,CloseDate,year,month,yrmo,ListingContractDate,PurchaseContractDate,listing_to_contract_days,contract_to_close_days
0,551985747,240000.0,295000.0,499000.0,1140.0,0.813559,0.480962,210.526316,777,2024-01-26,2024,1,2024-01-01,2021-10-06,2023-11-22,777.0,65.0
1,522107581,815000.0,759900.0,759900.0,1974.0,1.072510,1.072510,412.867275,33,2024-01-05,2024,1,2024-01-01,2021-03-08,2021-06-30,114.0,919.0
2,510919001,810000.0,770000.0,739900.0,1974.0,1.051948,1.094743,410.334347,228,2024-01-05,2024,1,2024-01-01,2021-03-08,2021-11-18,255.0,778.0
3,1079166779,858000.0,858000.0,NaN,1995.0,1.000000,NaN,430.075188,0,2024-01-30,2024,1,2024-01-01,2024-01-30,2024-08-05,188.0,-188.0
4,1075037759,1890500.0,1890500.0,1890500.0,3194.0,1.000000,1.000000,591.891046,0,2024-01-29,2024,1,2024-01-01,2024-01-29,2024-01-29,0.0,0.0
5,1067652762,2100000.0,2100000.0,2100000.0,3736.0,1.000000,1.000000,562.098501,0,2024-01-02,2024,1,2024-01-01,2023-11-15,2023-11-15,0.0,48.0
6,1061988701,2340000.0,2340000.0,NaN,2442.0,1.000000,NaN,958.230958,0,2024-01-31,2024,1,2024-01-01,2024-01-31,2024-03-04,33.0,-33.0
7,1061822266,1485000.0,1550000.0,1550000.0,1601.0,0.958065,0.958065,927.545284,0,2024-01-31,2024,1,2024-01-01,2024-01-09,2024-01-10,1.0,21.0
8,1061617422,1130000.0,999000.0,999000.0,2136.0,1.131131,1.131131,529.026217,1,2024-01-30,2024,1,2024-01-01,2024-01-12,2024-01-13,1.0,17.0
9,1060223286,1060000.0,1050000.0,1050000.0,1917.0,1.009524,1.009524,552.947314,34,2024-01-16,2024,1,2024-01-01,2023-12-07,2024-01-10,34.0,6.0


In [90]:
list_sample_columns = [
    "ListingKey",
    "ListPrice",
    "LivingArea",
    "days_on_market",
    "ListingContractDate",
    "year",
    "month",
    "yrmo",
    "list_price_per_sqft"
]

existing_list_sample_columns = [
    col
    for col in list_sample_columns
    if col in list_features.columns
]

display(
    list_features[
        existing_list_sample_columns
    ].head(10)
)

,ListingKey,ListPrice,LivingArea,days_on_market,ListingContractDate,year,month,yrmo,list_price_per_sqft
0,1074973329,1340000.0,1301.0,127,2024-01-01,2024,1,2024-01-01,1029.976941
1,1074954552,2500000.0,2788.0,1,2024-01-24,2024,1,2024-01-01,896.700143
2,1074936537,3150000.0,3250.0,1,2024-01-12,2024,1,2024-01-01,969.230769
3,1074917818,3090000.0,7456.0,0,2024-01-20,2024,1,2024-01-01,414.431330
4,1074143166,12725000.0,2321.0,2,2024-01-12,2024,1,2024-01-01,5482.550625
5,1073349180,665000.0,1487.0,2,2024-01-30,2024,1,2024-01-01,447.209146
6,1073322022,1125000.0,1750.0,2,2024-01-26,2024,1,2024-01-01,642.857143
7,1073315828,1849000.0,3152.0,3,2024-01-08,2024,1,2024-01-01,586.611675
8,1073313770,1389000.0,2282.0,5,2024-01-25,2024,1,2024-01-01,608.676599
9,1073295259,1249888.0,1899.0,0,2024-01-29,2024,1,2024-01-01,658.182201


### 7. Add School Districts to Sold and List

The California School District Areas dataset contains Elementary, High, and Unified school district boundaries. To follow the project requirements and avoid overlapping district types, the boundary dataset is filtered to include only `DistrictType = "Unified"`.

Each property's latitude and longitude values are converted into a geographic point. A spatial join is then performed to identify the Unified School District polygon that contains each property. The matched `DistrictName` is added to the Sold and List datasets as `UnifiedSchoolDistrict`.

#### Define School District Data File Path

In [22]:
# School district geographic data
GEOGRAPHIC_DATA_DIR = DATA_DIR / "school_district"

SCHOOL_DISTRICT_FILE = (
    GEOGRAPHIC_DATA_DIR
    / "california_school_district_areas_2025_26.geojson"
)

# Final enriched output files
SOLD_ENRICHED_FILE = (
    FEATURE_OUTPUT_DIR
    / "sold_residential_features_enriched.csv"
)

LIST_ENRICHED_FILE = (
    FEATURE_OUTPUT_DIR
    / "list_residential_features_enriched.csv"
)

In [23]:
print("Geographic data directory exists:", GEOGRAPHIC_DATA_DIR.exists())
print("School district file exists:", SCHOOL_DISTRICT_FILE.exists())
print("Feature output directory exists:", FEATURE_OUTPUT_DIR.exists())

#print("\nSchool district file:")
#print(SCHOOL_DISTRICT_FILE)

#print("\nSold enriched output:")
#print(SOLD_ENRICHED_FILE)

#print("\nList enriched output:")
#print(LIST_ENRICHED_FILE)

Geographic data directory exists: True
School district file exists: True
Feature output directory exists: True


In [24]:
# Confirm that the school district GeoJSON file exists
if not SCHOOL_DISTRICT_FILE.exists():
    raise FileNotFoundError(
        f"School district GeoJSON not found:\n"
        f"{SCHOOL_DISTRICT_FILE.resolve()}"
    )

print("School district file found:")
print(SCHOOL_DISTRICT_FILE)

School district file found:
cleaned_data/school_district/california_school_district_areas_2025_26.geojson


#### 7.1 Load California School District Boundary Data

Load the downloaded California School District boundary GeoJSON as a GeoDataFrame. 

The file contains geographic polygon boundaries that will be used to determine which Unified School District contains each property.

In [25]:
school_districts = gpd.read_file(
    SCHOOL_DISTRICT_FILE
)

print("School district shape:", school_districts.shape)
print("School district CRS:", school_districts.crs)

School district shape: (936, 51)
School district CRS: EPSG:3857


#### 7.2 Inspect and Validate School District Data

After loading the school district boundary data, the dataset structure was
reviewed to confirm the available columns, coordinate reference system (CRS),
district type categories, and geometry quality. 

This validation ensures that the required district name, district type, and geometry fields are available
before filtering the data and performing the spatial join.

In [ ]:
# Check dataset dimensions and CRS
print("School district shape:", school_districts.shape)
print("School district CRS:", school_districts.crs) #crs = Coordinate Reference System
print("Total columns:", len(school_districts.columns))

# Identify relevant columns
relevant_columns = [
    col
    for col in school_districts.columns
    if "district" in col.lower()
    or col == "geometry"
]

print("\nRelevant columns:")
for col in relevant_columns:
    print(f"- {col}")

# Preview relevant fields
display(
    school_districts[relevant_columns].head()
)

# Check geometry quality
print(
    "\nMissing geometry:",
    school_districts.geometry.isna().sum()
)

print(
    "Invalid geometry:",
    (~school_districts.geometry.is_valid).sum()
)

# Check DistrictType's unique values
print("\nDistrictType's unique values:")
display(
    school_districts["DistrictType"]
    .value_counts(dropna=False)
    .rename_axis("DistrictType")
    .reset_index(name="record_count")
)

School district shape: (936, 51)
School district CRS: EPSG:3857
Total columns: 51

Relevant columns:
- DistrictName
- DistrictType
- geometry


,DistrictName,DistrictType,geometry
0,Alameda Unified,Unified,"MULTIPOLYGON (((-13606222.82 4540862.699, -136..."
1,Albany City Unified,Unified,"POLYGON ((-13612893.866 4565099.707, -13612896..."
2,Berkeley Unified,Unified,"POLYGON ((-13609482.48 4565074.597, -13609483...."
3,Castro Valley Unified,Unified,"MULTIPOLYGON (((-13582508.535 4529067.071, -13..."
4,Emery Unified,Unified,"POLYGON ((-13613999.038 4555592.769, -13614126..."



Missing geometry: 0
Invalid geometry: 7

DistrictType's unique values:


,DistrictType,record_count
0,Elementary,515
1,Unified,345
2,High,76


#### 7.3 Filter Unified School Districts

In [27]:
# Filter the boundary data to Unified School Districts only

# Standardize district type values by removing extra spaces
# and ignoring capitalization differences before filtering
unified_districts = (
    school_districts.loc[
        school_districts["DistrictType"]
        .astype("string")
        .str.strip()
        .str.casefold()
        .eq("unified")
    ]
    .copy()
)

print(
    "Original school district records:",
    f"{len(school_districts):,}"
)

print(
    "Unified school district records:",
    f"{len(unified_districts):,}"
)

Original school district records: 936
Unified school district records: 345


In [28]:
# Keep only the columns needed for spatial mapping
unified_districts = unified_districts[
    [
        "DistrictName",
        "geometry"
    ]
].copy()

display(unified_districts.head())

,DistrictName,geometry
0,Alameda Unified,"MULTIPOLYGON (((-13606222.82 4540862.699, -136..."
1,Albany City Unified,"POLYGON ((-13612893.866 4565099.707, -13612896..."
2,Berkeley Unified,"POLYGON ((-13609482.48 4565074.597, -13609483...."
3,Castro Valley Unified,"MULTIPOLYGON (((-13582508.535 4529067.071, -13..."
4,Emery Unified,"POLYGON ((-13613999.038 4555592.769, -13614126..."


In [29]:
print("Unified district shape:", unified_districts.shape)
print("Unified district CRS:", unified_districts.crs)

print(
    "Missing district names:",
    unified_districts["DistrictName"].isna().sum()
)

print(
    "Missing geometries:",
    unified_districts.geometry.isna().sum()
)

print(
    "Invalid geometries:",
    (~unified_districts.geometry.is_valid).sum()
)

Unified district shape: (345, 2)
Unified district CRS: EPSG:3857
Missing district names: 0
Missing geometries: 0
Invalid geometries: 3


The school district boundary dataset contains multiple district types.

Only records classified as `Unified` were retained because the project
requires mapping properties to Unified School District boundaries. 

The filtered dataset was then reduced to the `DistrictName` and `geometry`
fields needed for the spatial join.

#### 7.4 Convert Property Coordinates to Geographic Points

Property latitude and longitude values were converted into geographic point
geometries so that each property could be spatially matched to a Unified
School District polygon. 

Only records with valid California coordinates were 
included in the GeoDataFrame. The property points were initially assigned 
the WGS 84 coordinate reference system (`EPSG:4326`), which is commonly used for
latitude and longitude data.

##### Sold Dataset

In [30]:
# Convert property coordinates to numeric values
sold_features["Latitude"] = pd.to_numeric(
    sold_features["Latitude"],
    errors="coerce"
)

sold_features["Longitude"] = pd.to_numeric(
    sold_features["Longitude"],
    errors="coerce"
)

In [31]:
# Identify records with usable California coordinates
sold_valid_coordinates = (
    sold_features["Latitude"].between(32, 42)
    & sold_features["Longitude"].between(-125, -114)
)

print(
    "Total Sold records:",
    f"{len(sold_features):,}"
)

print(
    "Sold records with valid coordinates:",
    f"{sold_valid_coordinates.sum():,}"
)

print(
    "Sold records excluded from spatial mapping:",
    f"{(~sold_valid_coordinates).sum():,}"
)

Total Sold records: 447,769
Sold records with valid coordinates: 431,453
Sold records excluded from spatial mapping: 16,316


In [32]:
# Create a subset containing records with valid coordinates
sold_coordinate_data = sold_features.loc[
    sold_valid_coordinates,
    [
        "Latitude",
        "Longitude"
    ]
].copy()

In [33]:
# Convert longitude and latitude into geographic point geometries
sold_points = gpd.GeoDataFrame(
    sold_coordinate_data,
    geometry=gpd.points_from_xy(
        sold_coordinate_data["Longitude"],
        sold_coordinate_data["Latitude"]
    ),
    crs="EPSG:4326"
)

In [34]:
# Validate after convertion
print("Sold points shape:", sold_points.shape)
print("Sold points CRS:", sold_points.crs)

display(
    sold_points[
        [
            "Latitude",
            "Longitude",
            "geometry"
        ]
    ].head()
)

Sold points shape: (431453, 3)
Sold points CRS: EPSG:4326


,Latitude,Longitude,geometry
4,34.164376,-118.558749,POINT (-118.55875 34.16438)
5,35.277199,-120.646786,POINT (-120.64679 35.2772)
7,32.715089,-117.170568,POINT (-117.17057 32.71509)
8,34.070215,-118.258618,POINT (-118.25862 34.07022)
9,34.107850,-118.219273,POINT (-118.21927 34.10785)


In [35]:
print("Property points CRS:", sold_points.crs)
print("Unified districts CRS:", unified_districts.crs)

Property points CRS: EPSG:4326
Unified districts CRS: EPSG:3857


In [36]:
# Match the property point CRS to the district boundary CRS
sold_points = sold_points.to_crs(
    unified_districts.crs
)

print(
    "Updated Sold points CRS:",
    sold_points.crs
)

Updated Sold points CRS: EPSG:3857


##### List Dataset

In [91]:
# Convert valid longitude and latitude values into point geometries
# and align their CRS with the school district boundaries

# Convert property coordinates to numeric values
list_features["Latitude"] = pd.to_numeric(
    list_features["Latitude"],
    errors="coerce"
)

list_features["Longitude"] = pd.to_numeric(
    list_features["Longitude"],
    errors="coerce"
)

# Identify records with usable California coordinates
list_valid_coordinates = (
    list_features["Latitude"].between(32, 42)
    & list_features["Longitude"].between(-125, -114)
)

# Create a subset containing records with valid coordinates
list_coordinate_data = list_features.loc[
    list_valid_coordinates,
    [
        "Latitude",
        "Longitude"
    ]
].copy()

# Convert longitude and latitude into geographic point geometries
list_points = gpd.GeoDataFrame(
    list_coordinate_data,
    geometry=gpd.points_from_xy(
        list_coordinate_data["Longitude"],
        list_coordinate_data["Latitude"]
    ),
    crs="EPSG:4326"
)

# Match the district boundary CRS
list_points = list_points.to_crs(
    unified_districts.crs
)

print("List points shape:", list_points.shape)
print("List points CRS:", list_points.crs)

display(list_points.head())

List points shape: (534016, 3)
List points CRS: EPSG:3857


,Latitude,Longitude,geometry
0,34.052207,-118.408445,POINT (-13181167.803 4035814.307)
1,33.496363,-117.691677,POINT (-13101377.554 3961374.707)
2,34.119345,-118.111254,POINT (-13148084.652 4044838.423)
3,33.984057,-117.802819,POINT (-13113749.825 4026661.472)
4,33.607583,-117.887743,POINT (-13123203.522 3976230.938)


#### 7.5 Perform Spatial Join

##### Sold Spatial Join
A spatial left join was performed to match each Sold property point to the
Unified School District polygon containing it. 

The matched `DistrictName` was
then added back to the complete Sold dataset, while unmatched records were
retained with a missing district value.

In [39]:
# Perform a spatial join to identify which Unified School District
# polygon contains each Sold property point
sold_joined = gpd.sjoin(
    
    # Left GeoDataFrame:
    # Sold property records represented as geographic points
    sold_points,
    
    # Right GeoDataFrame:
    # Keep only the district name and polygon geometry needed for matching
    unified_districts[
        [
            "DistrictName",
            "geometry"
        ]
    ],
    
    # Retain every Sold property point, including points that do not
    # match any Unified School District polygon
    how="left",
    
    # Match a property only when its point geometry is located
    # inside a Unified School District polygon
    predicate="within"
)

In [40]:
# Review the spatial join result
print("Sold points before join:", f"{len(sold_points):,}")
print("Sold rows after join:", f"{len(sold_joined):,}")

display(
    sold_joined[
        [
            "Latitude",
            "Longitude",
            "DistrictName",
            "geometry"
        ]
    ].head()
)

Sold points before join: 431,453
Sold rows after join: 431,453


,Latitude,Longitude,DistrictName,geometry
4,34.164376,-118.558749,Los Angeles Unified,POINT (-13197899.568 4050895.12)
5,35.277199,-120.646786,San Luis Coastal Unified,POINT (-13430338.783 4201615.479)
7,32.715089,-117.170568,San Diego Unified,POINT (-13043367.966 3857547.568)
8,34.070215,-118.258618,Los Angeles Unified,POINT (-13164489.138 4038234.086)
9,34.107850,-118.219273,Los Angeles Unified,POINT (-13160109.272 4043292.856)


In [41]:
# Count matched and unmatched Sold property points
sold_matched_count = (
    sold_joined["DistrictName"]
    .notna()
    .sum()
)

sold_unmatched_count = (
    sold_joined["DistrictName"]
    .isna()
    .sum()
)

print(
    "Matched Sold property points:",
    f"{sold_matched_count:,}"
)

print(
    "Unmatched Sold property points:",
    f"{sold_unmatched_count:,}"
)

print(
    "Sold point match rate:",
    f"{sold_matched_count / len(sold_joined):.2%}"
)

Matched Sold property points: 327,678
Unmatched Sold property points: 103,775
Sold point match rate: 75.95%


In [42]:
# Check whether any property point matched multiple districts
sold_duplicate_matches = (
    sold_joined.index
    .duplicated(keep=False)
)

print(
    "Sold properties with multiple joined rows:",
    f"{sold_joined.loc[sold_duplicate_matches].index.nunique():,}"
)

Sold properties with multiple joined rows: 0


In [43]:
# Confirm that the original Sold index is unique
if not sold_features.index.is_unique:
    raise ValueError(
        "sold_features index is not unique."
    )

In [44]:
# Create a district mapping using the original property index
sold_district_mapping = (
    sold_joined["DistrictName"]
)

In [45]:
# Add DistrictName back to the complete Sold dataset
sold_enriched = sold_features.copy()

sold_enriched["DistrictName"] = (
    sold_enriched.index
    .to_series()
    .map(sold_district_mapping)
    .astype("string")
)

In [94]:
# Validate the enriched Sold dataset
print(
    "Original Sold rows:",
    f"{len(sold_features):,}"
)

print(
    "Enriched Sold rows:",
    f"{len(sold_enriched):,}"
)

print(
    "DistrictName added:",
    "DistrictName" in sold_enriched.columns
)

print(
    "Matched districts:",
    f"{sold_enriched['DistrictName'].notna().sum():,}"
)

print(
    "Missing districts:",
    f"{sold_enriched['DistrictName'].isna().sum():,}"
)

# Check Sample geo data
display(
    sold_enriched[
        [
            "Latitude",
            "Longitude",
            "DistrictName"
        ]
    ].head(10)
)

Original Sold rows: 447,769
Enriched Sold rows: 447,769
DistrictName added: True
Matched districts: 327,678
Missing districts: 120,091


,Latitude,Longitude,DistrictName
0,NaN,NaN,<NA>
1,NaN,NaN,<NA>
2,NaN,NaN,<NA>
3,NaN,NaN,<NA>
4,34.164376,-118.558749,Los Angeles Unified
5,35.277199,-120.646786,San Luis Coastal Unified
6,NaN,NaN,<NA>
7,32.715089,-117.170568,San Diego Unified
8,34.070215,-118.258618,Los Angeles Unified
9,34.107850,-118.219273,Los Angeles Unified


##### List Spatial Join

A spatial left join was performed to match each List property point to the
Unified School District polygon containing it. 

The matched `DistrictName` was
then added back to the complete List dataset, while unmatched records were
retained with a missing district value.

In [48]:
# Spatially match each List property point to the Unified School District
# polygon that contains it. The left join retains all valid property points,
# including unmatched records.
list_joined = gpd.sjoin(
    list_points,
    unified_districts[
        ["DistrictName", "geometry"]
    ],
    how="left",
    predicate="within"
)

In [49]:
# Review the List spatial join results
print(
    "List points before join:",
    f"{len(list_points):,}"
)

print(
    "List rows after join:",
    f"{len(list_joined):,}"
)

print(
    "Matched List property points:",
    f"{list_joined['DistrictName'].notna().sum():,}"
)

print(
    "Unmatched List property points:",
    f"{list_joined['DistrictName'].isna().sum():,}"
)

display(
    list_joined[
        [
            "Latitude",
            "Longitude",
            "DistrictName",
            "geometry"
        ]
    ].head()
)

List points before join: 534,018
List rows after join: 534,018
Matched List property points: 411,516
Unmatched List property points: 122,502


,Latitude,Longitude,DistrictName,geometry
0,34.052207,-118.408445,Los Angeles Unified,POINT (-13181167.803 4035814.307)
1,33.496363,-117.691677,Capistrano Unified,POINT (-13101377.554 3961374.707)
2,34.119345,-118.111254,San Marino Unified,POINT (-13148084.652 4044838.423)
3,33.984057,-117.802819,Walnut Valley Unified,POINT (-13113749.825 4026661.472)
4,33.607583,-117.887743,Newport-Mesa Unified,POINT (-13123203.522 3976230.938)


In [50]:
# Check whether any List property matched multiple district polygons
list_duplicate_matches = (
    list_joined.index
    .duplicated(keep=False)
)

print(
    "List properties with multiple district matches:",
    f"{list_joined.loc[list_duplicate_matches].index.nunique():,}"
)

List properties with multiple district matches: 0


In [51]:
# Confirm that the original List dataset has a unique index
if not list_features.index.is_unique:
    raise ValueError(
        "list_features index is not unique."
    )

In [52]:
# Create a DistrictName mapping based on the original property index
list_district_mapping = list_joined["DistrictName"]

In [53]:
# Add DistrictName back to the complete List dataset
list_enriched = list_features.copy()

list_enriched["DistrictName"] = (
    list_enriched.index
    .to_series()
    .map(list_district_mapping)
    .astype("string")
)

In [54]:
# Validate the enriched List dataset
print(
    "Original List rows:",
    f"{len(list_features):,}"
)

print(
    "Enriched List rows:",
    f"{len(list_enriched):,}"
)

print(
    "Matched districts:",
    f"{list_enriched['DistrictName'].notna().sum():,}"
)

print(
    "Missing districts:",
    f"{list_enriched['DistrictName'].isna().sum():,}"
)

display(
    list_enriched[
        [
            "Latitude",
            "Longitude",
            "DistrictName"
        ]
    ].head(10)
)

Original List rows: 615,316
Enriched List rows: 615,316
Matched districts: 411,516
Missing districts: 203,800


,Latitude,Longitude,DistrictName
0,34.052207,-118.408445,Los Angeles Unified
1,33.496363,-117.691677,Capistrano Unified
2,34.119345,-118.111254,San Marino Unified
3,33.984057,-117.802819,Walnut Valley Unified
4,33.607583,-117.887743,Newport-Mesa Unified
5,32.929350,-116.945770,<NA>
6,33.761176,-116.419871,Palm Springs Unified
7,34.206316,-118.221526,La Canada Unified
8,33.862141,-117.723722,Orange Unified
9,33.477414,-117.664879,Capistrano Unified


### 8. Segment Analysis

The feature-engineered and school-district-enriched datasets were grouped by
key property, geographic, and competitive dimensions to identify market
patterns.

Sold records were used to calculate closed-sale metrics such as median close
price, sales volume, price per square foot, days on market, price ratios, and
transaction timeline measures. List records were used to summarize listing
activity by property type, geography, and listing office.

#### 8.1 Define Segment Dimensions and Metrics

In [55]:
# Define the dimensions used for segment analysis
property_segment_columns = [
    "PropertyType",
    "PropertySubType"
]

geographic_segment_columns = [
    "CountyOrParish",
    "MLSAreaMajor"
]

competitive_segment_columns = [
    "ListOfficeName",
    "BuyerOfficeName"
]

In [56]:
# Define the columns required for Sold segment analysis
required_sold_segment_columns = [
    "PropertyType",
    "PropertySubType",
    "CountyOrParish",
    "MLSAreaMajor",
    "ListOfficeName",
    "BuyerOfficeName",
    "ClosePrice",
    "price_per_sqft",
    "days_on_market",
    "price_ratio",
    "close_to_original_list_ratio",
    "listing_to_contract_days",
    "contract_to_close_days"
]

missing_sold_segment_columns = [
    col
    for col in required_sold_segment_columns
    if col not in sold_enriched.columns
]

if missing_sold_segment_columns:
    raise KeyError(
        "Missing Sold segment analysis columns: "
        f"{missing_sold_segment_columns}"
    )

print("All required Sold segment analysis columns are available.")

All required Sold segment analysis columns are available.


#### 8.2 Create Sold Property Segment Summary

In [57]:
def create_sold_segment_summary(df, group_columns):
    """
    Generate Sold market summary statistics for selected segment columns.
    """

    summary = (
        df.groupby(
            group_columns,
            dropna=False
        )
        .agg(
            closed_sales=(
                "ClosePrice",
                "size"
            ),
            total_sales_volume=(
                "ClosePrice",
                "sum"
            ),
            median_close_price=(
                "ClosePrice",
                "median"
            ),
            average_price_per_sqft=(
                "price_per_sqft",
                "mean"
            ),
            average_days_on_market=(
                "days_on_market",
                "mean"
            ),
            average_price_ratio=(
                "price_ratio",
                "mean"
            ),
            average_close_to_original_list_ratio=(
                "close_to_original_list_ratio",
                "mean"
            ),
            average_listing_to_contract_days=(
                "listing_to_contract_days",
                "mean"
            ),
            average_contract_to_close_days=(
                "contract_to_close_days",
                "mean"
            )
        )
        .reset_index()
    )

    return summary

In [58]:
# Summarize Sold market performance by property type and subtype
sold_property_segment_summary = create_sold_segment_summary(
    df=sold_enriched,
    group_columns=[
        "PropertyType",
        "PropertySubType"
    ]
)

In [59]:
sold_property_segment_summary = (
    sold_property_segment_summary
    .sort_values(
        by="closed_sales",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    sold_property_segment_summary.head(10)
)

,PropertyType,PropertySubType,closed_sales,total_sales_volume,median_close_price,average_price_per_sqft,average_days_on_market,average_price_ratio,average_close_to_original_list_ratio,average_listing_to_contract_days,average_contract_to_close_days
0,Residential,SingleFamilyResidence,335426,4.349177e+11,895000.0,637.318573,36.102210,1.061084,56.772791,44.190208,31.506132
1,Residential,Condominium,73587,6.445973e+10,627000.0,704.227307,41.935586,1.172134,24.057885,48.778447,30.873091
2,Residential,Townhouse,26240,2.647589e+10,800000.0,656.989357,32.277401,1.125475,38.539130,38.520758,30.958255
3,Residential,ManufacturedOnLand,5791,2.033528e+09,320000.0,245.974160,59.023830,0.973692,1.263969,69.810913,40.388707
4,Residential,Duplex,2485,3.029052e+09,910000.0,639.077684,42.557746,0.990026,392.630730,55.088934,38.204024
5,Residential,StockCooperative,1764,6.931456e+08,360000.0,407.689869,38.392857,0.988932,1.514161,48.799320,47.375283
6,Residential,NaN,860,8.456844e+08,801944.0,2141.485429,44.484884,1.001832,0.994669,45.290548,29.182030
7,Residential,Cabin,507,1.469952e+08,241000.0,592.666578,79.408284,0.937526,0.897006,86.481262,35.986193
8,Residential,Triplex,376,4.951903e+08,1131000.0,582.969857,55.864362,0.970621,1.020213,68.132979,44.659574
9,Residential,MixedUse,222,2.151269e+08,687500.0,496.630997,85.725225,0.945843,0.888445,97.954955,47.454955


Sold records were grouped by `PropertyType` and `PropertySubType` to compare
transaction volume, pricing, market speed, price performance, and transaction
timelines across different residential property segments.

#### 8.3 Create Sold Geographic Segment Summary

In [60]:
# Summarize Sold market performance by county and MLS area
sold_geographic_segment_summary = create_sold_segment_summary(
    df=sold_enriched,
    group_columns=[
        "CountyOrParish",
        "MLSAreaMajor"
    ]
)

In [61]:
sold_geographic_segment_summary = (
    sold_geographic_segment_summary
    .sort_values(
        by="closed_sales",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    sold_geographic_segment_summary.head(10)
)

,CountyOrParish,MLSAreaMajor,closed_sales,total_sales_volume,median_close_price,average_price_per_sqft,average_days_on_market,average_price_ratio,average_close_to_original_list_ratio,average_listing_to_contract_days,average_contract_to_close_days
0,Riverside,SRCAR - Southwest Riverside County,21860,1.397411e+10,587593.5,321.594756,41.436597,1.043603,67.702017,53.267566,37.053522
1,Contra Costa,NaN,19785,2.256376e+10,829000.0,585.654218,28.345312,1.071395,1.058079,29.655226,25.799788
2,Alameda,NaN,19274,2.537633e+10,1135000.0,773.114525,25.690308,1.126017,1.248063,26.714330,25.263983
3,Santa Clara,699 - Not Defined,18221,3.555065e+10,1610000.0,1076.532038,21.482630,1.048432,1.040847,21.504638,25.298611
4,San Mateo,699 - Not Defined,7307,1.640368e+10,1735000.0,1126.680107,27.658547,1.043031,1.032209,22.193650,29.009443
5,Riverside,252 - Riverside,5670,4.038801e+09,657250.0,390.008892,37.903704,1.000719,1.018354,47.956261,36.567019
6,Butte,NaN,4400,1.913982e+09,400000.0,261.940496,52.008636,0.990331,1.199782,59.933409,34.481364
7,Monterey,699 - Not Defined,4158,5.859198e+09,910580.0,753.059162,45.284031,0.985809,0.964626,47.248918,30.938191
8,Riverside,248 - Corona,3603,2.885012e+09,759000.0,403.691305,39.400500,0.997526,285.997166,50.090758,35.099917
9,Los Angeles,LAC - Lancaster,3324,1.628106e+09,475000.0,277.674351,41.373947,1.003666,1.588738,48.835138,38.527677


Sold records were grouped by `CountyOrParish` and `MLSAreaMajor` to identify
differences in transaction volume, home prices, market speed, and price
performance across geographic market segments.

In [62]:
# Summarize Sold market performance by school district
sold_school_district_summary = create_sold_segment_summary(
    df=sold_enriched,
    group_columns=["DistrictName"]
)

display(
    sold_school_district_summary.head(10)
)

,DistrictName,closed_sales,total_sales_volume,median_close_price,average_price_per_sqft,average_days_on_market,average_price_ratio,average_close_to_original_list_ratio,average_listing_to_contract_days,average_contract_to_close_days
0,ABC Unified,1041,9.879895e+08,900000.0,631.849931,24.232469,1.010236,97.095103,34.910663,31.336215
1,Acton-Agua Dulce Unified,316,2.756634e+08,835570.0,394.016841,62.810127,0.983191,0.963406,71.677215,40.503165
2,Alameda Unified,907,1.101418e+09,1170000.0,728.540380,24.339581,1.092762,1.084727,25.114664,23.970232
3,Albany City Unified,228,2.821775e+08,1185000.0,896.715197,27.100877,1.194412,1.201019,28.320175,20.964912
4,Alhambra Unified,1370,1.284409e+09,880000.0,637.653378,31.516058,1.020863,1.667504,40.081752,30.795620
5,Alpaugh Unified,1,1.900000e+05,190000.0,227.272727,22.000000,0.950000,0.950000,22.000000,37.000000
6,Alpine County Unified,2,1.899000e+06,949500.0,389.158189,144.000000,0.924710,0.833333,144.000000,64.000000
7,Alvord Unified,1249,8.472995e+08,650000.0,403.961093,35.509207,1.001831,0.994859,45.327462,35.117694
8,Amador County Unified,21,8.858136e+06,425000.0,221.204302,130.190476,0.951261,0.874922,132.904762,27.952381
9,Anderson Valley Unified,6,4.981849e+06,731000.0,364.251615,108.666667,0.980979,0.985332,131.666667,36.333333


#### 8.4 Create Competitive Office Summaries

In [63]:
# Summarize closed-sale performance by listing office
listing_office_summary = create_sold_segment_summary(
    df=sold_enriched,
    group_columns=["ListOfficeName"]
)

In [64]:
listing_office_summary = (
    listing_office_summary
    .sort_values(
        by=[
            "total_sales_volume",
            "closed_sales"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

display(
    listing_office_summary.head(10)
)

,ListOfficeName,closed_sales,total_sales_volume,median_close_price,average_price_per_sqft,average_days_on_market,average_price_ratio,average_close_to_original_list_ratio,average_listing_to_contract_days,average_contract_to_close_days
0,Compass,31717,5.875266e+10,1350000.0,868.928764,30.659426,1.055218,1.129004,34.775915,27.784503
1,Coldwell Banker Realty,20197,3.355254e+10,1200000.0,849.415326,34.534287,1.104398,5.934351,41.877861,27.123427
2,Keller Williams Realty,8759,9.051790e+09,875000.0,568.826346,31.519808,1.008404,1.533303,40.113255,30.601781
3,Berkshire Hathaway HomeServices California Pro...,5813,8.747033e+09,950000.0,701.023125,41.650611,0.994020,0.980224,48.782983,29.749397
4,Intero Real Estate Services,4808,7.963182e+09,1377950.0,937.075078,24.655990,1.042946,3.382672,24.700499,26.131448
5,Coldwell Banker West,2876,7.121547e+09,825000.0,1812.412097,28.267038,3.094962,3.065508,38.274339,28.609527
6,First Team Real Estate,6249,7.045552e+09,960000.0,619.879691,29.540887,1.005647,97.301984,44.068971,30.629541
7,The Agency,2941,6.827176e+09,1680000.0,888.554040,35.456987,1.014694,1.003008,38.908285,26.760655
8,Real Broker,5176,5.453011e+09,849900.0,586.092624,29.412287,1.004521,194.197435,40.716577,30.086360
9,Pacific Sotheby's Int'l Realty,1791,5.370119e+09,1675000.0,1286.439276,39.026801,1.533072,1.499723,50.268565,28.932440


In [65]:
# Summarize closed-sale performance by buyer office
buyer_office_summary = create_sold_segment_summary(
    df=sold_enriched,
    group_columns=["BuyerOfficeName"]
)

In [66]:
buyer_office_summary = (
    buyer_office_summary
    .sort_values(
        by=[
            "total_sales_volume",
            "closed_sales"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

display(
    buyer_office_summary.head(10)
)

,BuyerOfficeName,closed_sales,total_sales_volume,median_close_price,average_price_per_sqft,average_days_on_market,average_price_ratio,average_close_to_original_list_ratio,average_listing_to_contract_days,average_contract_to_close_days
0,Compass,29602,5.397955e+10,1338000.0,869.656862,32.011756,1.054877,1.549973,37.628234,27.148950
1,Coldwell Banker Realty,16233,2.780711e+10,1199000.0,866.380245,34.704121,1.139759,1.375463,41.854228,27.451861
2,NaN,7152,9.434752e+09,1098000.0,855.277164,32.323826,1.031741,1.018607,28.250734,33.708852
3,Keller Williams Realty,6901,8.618636e+09,840000.0,776.772997,34.721635,1.303063,1.689836,43.178406,30.008261
4,Real Broker,6751,6.907365e+09,803000.0,586.408522,33.252259,1.017903,1.152376,43.608593,30.059852
5,Berkshire Hathaway HomeServices California Pro...,4514,6.792172e+09,983500.0,704.706150,39.500443,0.996910,1.194165,47.462716,29.803817
6,NONMEMBER MRML,9767,6.212295e+09,501990.0,359.020551,48.051705,0.995812,1.788871,57.295587,37.134637
7,First Team Real Estate,5667,6.187015e+09,900000.0,614.817638,31.024528,1.007978,173.162587,42.313504,30.473080
8,Intero Real Estate Services,3880,6.144827e+09,1335000.0,891.376298,25.055155,1.037324,1.026333,25.329209,26.346997
9,The Agency,2746,6.140777e+09,1550000.0,862.045827,38.506555,1.006591,0.997294,43.816588,29.129339


Closed-sale records were grouped separately by `ListOfficeName` and
`BuyerOfficeName` to evaluate office-level sales volume, transaction units,
pricing, market speed, and price performance. These summaries support the
competitive analysis dashboards.

#### 8.5 Create List Segment Summaries

In [67]:
def create_list_segment_summary(df, group_columns):
    """
    Generate listing activity summary statistics
    for selected segment columns.
    """

    summary = (
        df.groupby(
            group_columns,
            dropna=False
        )
        .agg(
            listing_count=(
                "ListPrice",
                "size"
            ),
            median_list_price=(
                "ListPrice",
                "median"
            ),
            average_price_per_sqft=(
                "list_price_per_sqft",
                "mean"
            ),
            average_days_on_market=(
                "days_on_market",
                "mean"
            )
        )
        .reset_index()
    )

    return summary

In [68]:
# Property Segment
list_property_segment_summary = create_list_segment_summary(
    df=list_enriched,
    group_columns=[
        "PropertyType",
        "PropertySubType"
    ]
)

list_property_segment_summary = (
    list_property_segment_summary
    .sort_values(
        by="listing_count",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    list_property_segment_summary.head(10)
)

,PropertyType,PropertySubType,listing_count,median_list_price,average_price_per_sqft,average_days_on_market
0,Residential,SingleFamilyResidence,448132,928000.0,657.638886,17.905586
1,Residential,Condominium,111639,639990.0,643.660352,20.128297
2,Residential,Townhouse,36277,819000.0,618.153796,18.439066
3,Residential,ManufacturedOnLand,8487,329951.0,290.700133,19.571109
4,Residential,Duplex,3906,975000.0,678.147955,17.612391
5,Residential,StockCooperative,1905,375000.0,430.933535,18.607874
6,Residential,NaN,1413,814600.0,4863.853177,19.061571
7,Residential,Cabin,1138,299999.0,512.729780,21.113357
8,Residential,Triplex,787,1200000.0,667.131578,18.560356
9,Residential,MixedUse,572,850000.0,694.065886,21.554196


In [95]:
# Geographic Segment
list_geographic_segment_summary = create_list_segment_summary(
    df=list_enriched,
    group_columns=[
        "CountyOrParish",
        "MLSAreaMajor"
    ]
)


display(
    list_geographic_segment_summary.head(10)
)

,CountyOrParish,MLSAreaMajor,listing_count,median_list_price,average_price_per_sqft,average_days_on_market
0,Alameda,699 - Not Defined,3570,1098000.0,738.504120,18.254342
1,Alameda,BERK - Berkeley,6,795000.0,911.474774,26.833333
2,Alameda,GLV - Glenview,1,599000.0,378.156566,20.000000
3,Alameda,"RO - Compton S of Rosecrans, E of Alameda",1,325000.0,584.532374,1.000000
4,Alameda,VTU - Ventura,2,720000.0,698.351115,18.000000
5,Alameda,NaN,25870,998000.0,680.189829,15.981871
6,Alpine,699 - Not Defined,1,1100000.0,427.350427,0.000000
7,Alpine,NaN,1,16250.0,20.940722,0.000000
8,Amador,699 - Not Defined,21,510000.0,277.837243,20.523810
9,Amador,NaN,44,507500.0,299.060116,29.727273


In [96]:
# Listing Office Inventory
list_office_summary = create_list_segment_summary(
    df=list_enriched,
    group_columns=["ListOfficeName"]
)

list_office_summary = (
    list_office_summary
    .sort_values(
        by="listing_count",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    list_office_summary.head(10)
)

,ListOfficeName,listing_count,median_list_price,average_price_per_sqft,average_days_on_market
0,Compass,44187,1375000.0,888.346898,17.204925
1,Coldwell Banker Realty,27134,1225000.0,821.127419,18.733102
2,Keller Williams Realty,10636,875000.0,572.443917,16.397518
3,Berkshire Hathaway HomeServices California Pro...,7831,1035000.0,1009.182547,19.781254
4,Real Broker,7052,849999.0,598.409971,16.536302
5,Intero Real Estate Services,7047,1300000.0,884.712388,16.405279
6,First Team Real Estate,6804,949000.0,628.338194,16.569224
7,eXp Realty of California Inc,6794,799000.0,596.507131,19.194731
8,Equity Union,6061,895000.0,540.084748,22.059231
9,"eXp Realty of California, Inc.",5150,764450.0,556.978017,14.930097


#### 8.6 Validate and Save Segment Summaries

The segment summary tables were validated to confirm that the expected
grouping dimensions and calculated metrics were present. 

Duplicate segment
combinations and output dimensions were reviewed before each summary table
was saved as a CSV file for reporting and further analysis.

In [71]:
# Create a separate folder for segment analysis outputs
SEGMENT_OUTPUT_DIR = DATA_DIR / "segment_analysis"

SEGMENT_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Segment output directory:")
print(SEGMENT_OUTPUT_DIR)

Segment output directory:
cleaned_data/segment_analysis


In [72]:
# Collect all segment summary tables
segment_summaries = {
    "sold_property_segment_summary":
        sold_property_segment_summary,

    "sold_geographic_segment_summary":
        sold_geographic_segment_summary,

    "listing_office_summary":
        listing_office_summary,

    "buyer_office_summary":
        buyer_office_summary,

    "list_property_segment_summary":
        list_property_segment_summary,

    "list_geographic_segment_summary":
        list_geographic_segment_summary,

    "list_office_summary":
        list_office_summary
}

In [73]:
# Validate the size and missing values of each summary table
segment_validation_results = []

for summary_name, summary_df in segment_summaries.items():
    segment_validation_results.append({
        "summary_name": summary_name,
        "rows": len(summary_df),
        "columns": len(summary_df.columns),
        "fully_duplicated_rows": (
            summary_df.duplicated().sum()
        ),
        "total_missing_values": (
            summary_df.isna().sum().sum()
        )
    })

segment_validation_df = pd.DataFrame(
    segment_validation_results
)

display(segment_validation_df)

,summary_name,rows,columns,fully_duplicated_rows,total_missing_values
0,sold_property_segment_summary,21,11,0,1
1,sold_geographic_segment_summary,1303,11,0,62
2,listing_office_summary,19158,10,0,22
3,buyer_office_summary,21870,10,0,18
4,list_property_segment_summary,22,6,0,1
5,list_geographic_segment_summary,1386,6,0,67
6,list_office_summary,21586,5,0,4


In [74]:
# Define the grouping columns expected in each summary
segment_group_columns = {
    "sold_property_segment_summary": [
        "PropertyType",
        "PropertySubType"
    ],
    "sold_geographic_segment_summary": [
        "CountyOrParish",
        "MLSAreaMajor"
    ],
    "listing_office_summary": [
        "ListOfficeName"
    ],
    "buyer_office_summary": [
        "BuyerOfficeName"
    ],
    "list_property_segment_summary": [
        "PropertyType",
        "PropertySubType"
    ],
    "list_geographic_segment_summary": [
        "CountyOrParish",
        "MLSAreaMajor"
    ],
    "list_office_summary": [
        "ListOfficeName"
    ]
}

In [ ]:
# Confirm that each grouping combination appears only once (no duplicate)
for summary_name, group_columns in segment_group_columns.items():
    summary_df = segment_summaries[summary_name]

    duplicate_segments = summary_df.duplicated(
        subset=group_columns
    ).sum()

    print(
        f"{summary_name}: "
        f"{duplicate_segments:,} duplicate segments"
    )

sold_property_segment_summary: 0 duplicate segments
sold_geographic_segment_summary: 0 duplicate segments
listing_office_summary: 0 duplicate segments
buyer_office_summary: 0 duplicate segments
list_property_segment_summary: 0 duplicate segments
list_geographic_segment_summary: 0 duplicate segments
list_office_summary: 0 duplicate segments


In [76]:
# Confirm that every summary table contains records
for summary_name, summary_df in segment_summaries.items():
    assert not summary_df.empty, (
        f"{summary_name} is empty."
    )

print("All segment summary tables contain records.")

All segment summary tables contain records.


In [77]:
# Save each segment summary as a separate CSV file
for summary_name, summary_df in segment_summaries.items():
    output_file = (
        SEGMENT_OUTPUT_DIR
        / f"{summary_name}.csv"
    )

    summary_df.to_csv(
        output_file,
        index=False
    )

    print(f"Saved: {output_file}")

Saved: cleaned_data/segment_analysis/sold_property_segment_summary.csv
Saved: cleaned_data/segment_analysis/sold_geographic_segment_summary.csv
Saved: cleaned_data/segment_analysis/listing_office_summary.csv
Saved: cleaned_data/segment_analysis/buyer_office_summary.csv
Saved: cleaned_data/segment_analysis/list_property_segment_summary.csv
Saved: cleaned_data/segment_analysis/list_geographic_segment_summary.csv
Saved: cleaned_data/segment_analysis/list_office_summary.csv


### 9. Display Sample Outputs

Representative samples from the enriched property datasets and segment
summary tables were displayed to confirm that the engineered features,
school district mappings, grouping dimensions, and calculated market metrics
were generated correctly.

In [78]:
# Display selected columns from the enriched Sold dataset
sold_sample_columns = [
    "PropertyType",
    "PropertySubType",
    "CountyOrParish",
    "MLSAreaMajor",
    "ClosePrice",
    "price_per_sqft",
    "days_on_market",
    "DistrictName"
]

display(
    sold_enriched[sold_sample_columns].head(10)
)

,PropertyType,PropertySubType,CountyOrParish,MLSAreaMajor,ClosePrice,price_per_sqft,days_on_market,DistrictName
0,Residential,Condominium,San Mateo,699 - Not Defined,240000.0,210.526316,777,<NA>
1,Residential,SingleFamilyResidence,San Diego,91950 - National City,815000.0,412.867275,33,<NA>
2,Residential,SingleFamilyResidence,San Diego,91950 - National City,810000.0,410.334347,228,<NA>
3,Residential,SingleFamilyResidence,Riverside,NaN,858000.0,430.075188,0,<NA>
4,Residential,SingleFamilyResidence,Los Angeles,TAR - Tarzana,1890500.0,591.891046,0,Los Angeles Unified
5,Residential,SingleFamilyResidence,San Luis Obispo,SLO - San Luis Obispo,2100000.0,562.098501,0,San Luis Coastal Unified
6,Residential,SingleFamilyResidence,Santa Clara,699 - Not Defined,2340000.0,958.230958,0,<NA>
7,Residential,Condominium,San Diego,92101 - San Diego Downtown,1485000.0,927.545284,0,San Diego Unified
8,Residential,Duplex,Los Angeles,C21 - Silver Lake - Echo Park,1130000.0,529.026217,1,Los Angeles Unified
9,Residential,SingleFamilyResidence,Los Angeles,680 - Mount Washington,1060000.0,552.947314,34,Los Angeles Unified


In [79]:
# Display selected columns from the enriched List dataset
list_sample_columns = [
    "PropertyType",
    "PropertySubType",
    "CountyOrParish",
    "MLSAreaMajor",
    "ListPrice",
    "list_price_per_sqft",
    "days_on_market",
    "DistrictName"
]

display(
    list_enriched[list_sample_columns].head(10)
)

,PropertyType,PropertySubType,CountyOrParish,MLSAreaMajor,ListPrice,list_price_per_sqft,days_on_market,DistrictName
0,Residential,Condominium,Los Angeles,C05 - Westwood - Century City,1340000.0,1029.976941,127,Los Angeles Unified
1,Residential,SingleFamilyResidence,Orange,LNSLT - Salt Creek,2500000.0,896.700143,1,Capistrano Unified
2,Residential,SingleFamilyResidence,Los Angeles,655 - San Marino,3150000.0,969.230769,1,San Marino Unified
3,Residential,SingleFamilyResidence,Los Angeles,616 - Diamond Bar,3090000.0,414.431330,0,Walnut Valley Unified
4,Residential,SingleFamilyResidence,Orange,N9 - Lower Newport Bay - Balboa Island,12725000.0,5482.550625,2,Newport-Mesa Unified
5,Residential,ManufacturedOnLand,San Diego,92040 - Lakeside,665000.0,447.209146,2,<NA>
6,Residential,SingleFamilyResidence,Riverside,321 - Rancho Mirage,1125000.0,642.857143,2,Palm Springs Unified
7,Residential,SingleFamilyResidence,Los Angeles,634 - La Canada Flintridge,1849000.0,586.611675,3,La Canada Unified
8,Residential,SingleFamilyResidence,Orange,77 - Anaheim Hills,1389000.0,608.676599,5,Orange Unified
9,Residential,Townhouse,Orange,JS - San Juan South,1249888.0,658.182201,0,Capistrano Unified


In [80]:
# Display the largest Sold property segments by closed sales
display(
    sold_property_segment_summary.head(10)
)

,PropertyType,PropertySubType,closed_sales,total_sales_volume,median_close_price,average_price_per_sqft,average_days_on_market,average_price_ratio,average_close_to_original_list_ratio,average_listing_to_contract_days,average_contract_to_close_days
0,Residential,SingleFamilyResidence,335426,4.349177e+11,895000.0,637.318573,36.102210,1.061084,56.772791,44.190208,31.506132
1,Residential,Condominium,73587,6.445973e+10,627000.0,704.227307,41.935586,1.172134,24.057885,48.778447,30.873091
2,Residential,Townhouse,26240,2.647589e+10,800000.0,656.989357,32.277401,1.125475,38.539130,38.520758,30.958255
3,Residential,ManufacturedOnLand,5791,2.033528e+09,320000.0,245.974160,59.023830,0.973692,1.263969,69.810913,40.388707
4,Residential,Duplex,2485,3.029052e+09,910000.0,639.077684,42.557746,0.990026,392.630730,55.088934,38.204024
5,Residential,StockCooperative,1764,6.931456e+08,360000.0,407.689869,38.392857,0.988932,1.514161,48.799320,47.375283
6,Residential,NaN,860,8.456844e+08,801944.0,2141.485429,44.484884,1.001832,0.994669,45.290548,29.182030
7,Residential,Cabin,507,1.469952e+08,241000.0,592.666578,79.408284,0.937526,0.897006,86.481262,35.986193
8,Residential,Triplex,376,4.951903e+08,1131000.0,582.969857,55.864362,0.970621,1.020213,68.132979,44.659574
9,Residential,MixedUse,222,2.151269e+08,687500.0,496.630997,85.725225,0.945843,0.888445,97.954955,47.454955


In [81]:
# Display the largest Sold geographic segments by closed sales
display(
    sold_geographic_segment_summary.head(10)
)

,CountyOrParish,MLSAreaMajor,closed_sales,total_sales_volume,median_close_price,average_price_per_sqft,average_days_on_market,average_price_ratio,average_close_to_original_list_ratio,average_listing_to_contract_days,average_contract_to_close_days
0,Riverside,SRCAR - Southwest Riverside County,21860,1.397411e+10,587593.5,321.594756,41.436597,1.043603,67.702017,53.267566,37.053522
1,Contra Costa,NaN,19785,2.256376e+10,829000.0,585.654218,28.345312,1.071395,1.058079,29.655226,25.799788
2,Alameda,NaN,19274,2.537633e+10,1135000.0,773.114525,25.690308,1.126017,1.248063,26.714330,25.263983
3,Santa Clara,699 - Not Defined,18221,3.555065e+10,1610000.0,1076.532038,21.482630,1.048432,1.040847,21.504638,25.298611
4,San Mateo,699 - Not Defined,7307,1.640368e+10,1735000.0,1126.680107,27.658547,1.043031,1.032209,22.193650,29.009443
5,Riverside,252 - Riverside,5670,4.038801e+09,657250.0,390.008892,37.903704,1.000719,1.018354,47.956261,36.567019
6,Butte,NaN,4400,1.913982e+09,400000.0,261.940496,52.008636,0.990331,1.199782,59.933409,34.481364
7,Monterey,699 - Not Defined,4158,5.859198e+09,910580.0,753.059162,45.284031,0.985809,0.964626,47.248918,30.938191
8,Riverside,248 - Corona,3603,2.885012e+09,759000.0,403.691305,39.400500,0.997526,285.997166,50.090758,35.099917
9,Los Angeles,LAC - Lancaster,3324,1.628106e+09,475000.0,277.674351,41.373947,1.003666,1.588738,48.835138,38.527677


In [82]:
# Display the leading listing offices by sales volume
display(
    listing_office_summary.head(10)
)

,ListOfficeName,closed_sales,total_sales_volume,median_close_price,average_price_per_sqft,average_days_on_market,average_price_ratio,average_close_to_original_list_ratio,average_listing_to_contract_days,average_contract_to_close_days
0,Compass,31717,5.875266e+10,1350000.0,868.928764,30.659426,1.055218,1.129004,34.775915,27.784503
1,Coldwell Banker Realty,20197,3.355254e+10,1200000.0,849.415326,34.534287,1.104398,5.934351,41.877861,27.123427
2,Keller Williams Realty,8759,9.051790e+09,875000.0,568.826346,31.519808,1.008404,1.533303,40.113255,30.601781
3,Berkshire Hathaway HomeServices California Pro...,5813,8.747033e+09,950000.0,701.023125,41.650611,0.994020,0.980224,48.782983,29.749397
4,Intero Real Estate Services,4808,7.963182e+09,1377950.0,937.075078,24.655990,1.042946,3.382672,24.700499,26.131448
5,Coldwell Banker West,2876,7.121547e+09,825000.0,1812.412097,28.267038,3.094962,3.065508,38.274339,28.609527
6,First Team Real Estate,6249,7.045552e+09,960000.0,619.879691,29.540887,1.005647,97.301984,44.068971,30.629541
7,The Agency,2941,6.827176e+09,1680000.0,888.554040,35.456987,1.014694,1.003008,38.908285,26.760655
8,Real Broker,5176,5.453011e+09,849900.0,586.092624,29.412287,1.004521,194.197435,40.716577,30.086360
9,Pacific Sotheby's Int'l Realty,1791,5.370119e+09,1675000.0,1286.439276,39.026801,1.533072,1.499723,50.268565,28.932440


In [83]:
# Display the leading buyer offices by sales volume
display(
    buyer_office_summary.head(10)
)

,BuyerOfficeName,closed_sales,total_sales_volume,median_close_price,average_price_per_sqft,average_days_on_market,average_price_ratio,average_close_to_original_list_ratio,average_listing_to_contract_days,average_contract_to_close_days
0,Compass,29602,5.397955e+10,1338000.0,869.656862,32.011756,1.054877,1.549973,37.628234,27.148950
1,Coldwell Banker Realty,16233,2.780711e+10,1199000.0,866.380245,34.704121,1.139759,1.375463,41.854228,27.451861
2,NaN,7152,9.434752e+09,1098000.0,855.277164,32.323826,1.031741,1.018607,28.250734,33.708852
3,Keller Williams Realty,6901,8.618636e+09,840000.0,776.772997,34.721635,1.303063,1.689836,43.178406,30.008261
4,Real Broker,6751,6.907365e+09,803000.0,586.408522,33.252259,1.017903,1.152376,43.608593,30.059852
5,Berkshire Hathaway HomeServices California Pro...,4514,6.792172e+09,983500.0,704.706150,39.500443,0.996910,1.194165,47.462716,29.803817
6,NONMEMBER MRML,9767,6.212295e+09,501990.0,359.020551,48.051705,0.995812,1.788871,57.295587,37.134637
7,First Team Real Estate,5667,6.187015e+09,900000.0,614.817638,31.024528,1.007978,173.162587,42.313504,30.473080
8,Intero Real Estate Services,3880,6.144827e+09,1335000.0,891.376298,25.055155,1.037324,1.026333,25.329209,26.346997
9,The Agency,2746,6.140777e+09,1550000.0,862.045827,38.506555,1.006591,0.997294,43.816588,29.129339


In [84]:
print("Sample Enriched Sold Records")
display(
    sold_enriched[sold_sample_columns].head(10)
)

print("Sample Enriched List Records")
display(
    list_enriched[list_sample_columns].head(10)
)

print("Top Sold Property Segments")
display(
    sold_property_segment_summary.head(10)
)

print("Top Sold Geographic Segments")
display(
    sold_geographic_segment_summary.head(10)
)

print("Top Listing Offices")
display(
    listing_office_summary.head(10)
)

print("Top Buyer Offices")
display(
    buyer_office_summary.head(10)
)

Sample Enriched Sold Records


,PropertyType,PropertySubType,CountyOrParish,MLSAreaMajor,ClosePrice,price_per_sqft,days_on_market,DistrictName
0,Residential,Condominium,San Mateo,699 - Not Defined,240000.0,210.526316,777,<NA>
1,Residential,SingleFamilyResidence,San Diego,91950 - National City,815000.0,412.867275,33,<NA>
2,Residential,SingleFamilyResidence,San Diego,91950 - National City,810000.0,410.334347,228,<NA>
3,Residential,SingleFamilyResidence,Riverside,NaN,858000.0,430.075188,0,<NA>
4,Residential,SingleFamilyResidence,Los Angeles,TAR - Tarzana,1890500.0,591.891046,0,Los Angeles Unified
5,Residential,SingleFamilyResidence,San Luis Obispo,SLO - San Luis Obispo,2100000.0,562.098501,0,San Luis Coastal Unified
6,Residential,SingleFamilyResidence,Santa Clara,699 - Not Defined,2340000.0,958.230958,0,<NA>
7,Residential,Condominium,San Diego,92101 - San Diego Downtown,1485000.0,927.545284,0,San Diego Unified
8,Residential,Duplex,Los Angeles,C21 - Silver Lake - Echo Park,1130000.0,529.026217,1,Los Angeles Unified
9,Residential,SingleFamilyResidence,Los Angeles,680 - Mount Washington,1060000.0,552.947314,34,Los Angeles Unified


Sample Enriched List Records


,PropertyType,PropertySubType,CountyOrParish,MLSAreaMajor,ListPrice,list_price_per_sqft,days_on_market,DistrictName
0,Residential,Condominium,Los Angeles,C05 - Westwood - Century City,1340000.0,1029.976941,127,Los Angeles Unified
1,Residential,SingleFamilyResidence,Orange,LNSLT - Salt Creek,2500000.0,896.700143,1,Capistrano Unified
2,Residential,SingleFamilyResidence,Los Angeles,655 - San Marino,3150000.0,969.230769,1,San Marino Unified
3,Residential,SingleFamilyResidence,Los Angeles,616 - Diamond Bar,3090000.0,414.431330,0,Walnut Valley Unified
4,Residential,SingleFamilyResidence,Orange,N9 - Lower Newport Bay - Balboa Island,12725000.0,5482.550625,2,Newport-Mesa Unified
5,Residential,ManufacturedOnLand,San Diego,92040 - Lakeside,665000.0,447.209146,2,<NA>
6,Residential,SingleFamilyResidence,Riverside,321 - Rancho Mirage,1125000.0,642.857143,2,Palm Springs Unified
7,Residential,SingleFamilyResidence,Los Angeles,634 - La Canada Flintridge,1849000.0,586.611675,3,La Canada Unified
8,Residential,SingleFamilyResidence,Orange,77 - Anaheim Hills,1389000.0,608.676599,5,Orange Unified
9,Residential,Townhouse,Orange,JS - San Juan South,1249888.0,658.182201,0,Capistrano Unified


Top Sold Property Segments


...

,CountyOrParish,MLSAreaMajor,closed_sales,total_sales_volume,median_close_price,average_price_per_sqft,average_days_on_market,average_price_ratio,average_close_to_original_list_ratio,average_listing_to_contract_days,average_contract_to_close_days
0,Riverside,SRCAR - Southwest Riverside County,21860,1.397411e+10,587593.5,321.594756,41.436597,1.043603,67.702017,53.267566,37.053522
1,Contra Costa,NaN,19785,2.256376e+10,829000.0,585.654218,28.345312,1.071395,1.058079,29.655226,25.799788
2,Alameda,NaN,19274,2.537633e+10,1135000.0,773.114525,25.690308,1.126017,1.248063,26.714330,25.263983
3,Santa Clara,699 - Not Defined,18221,3.555065e+10,1610000.0,1076.532038,21.482630,1.048432,1.040847,21.504638,25.298611
4,San Mateo,699 - Not Defined,7307,1.640368e+10,1735000.0,1126.680107,27.658547,1.043031,1.032209,22.193650,29.009443
5,Riverside,252 - Riverside,5670,4.038801e+09,657250.0,390.008892,37.903704,1.000719,1.018354,47.956261,36.567019
6,Butte,NaN,4400,1.913982e+09,400000.0,261.940496,52.008636,0.990331,1.199782,59.933409,34.481364
7,Monterey,699 - Not Defined,4158,5.859198e+09,910580.0,753.059162,45.284031,0.985809,0.964626,47.248918,30.938191
8,Riverside,248 - Corona,3603,2.885012e+09,759000.0,403.691305,39.400500,0.997526,285.997166,50.090758,35.099917
9,Los Angeles,LAC - Lancaster,3324,1.628106e+09,475000.0,277.674351,41.373947,1.003666,1.588738,48.835138,38.527677


Top Listing Offices


,ListOfficeName,closed_sales,total_sales_volume,median_close_price,average_price_per_sqft,average_days_on_market,average_price_ratio,average_close_to_original_list_ratio,average_listing_to_contract_days,average_contract_to_close_days
0,Compass,31717,5.875266e+10,1350000.0,868.928764,30.659426,1.055218,1.129004,34.775915,27.784503
1,Coldwell Banker Realty,20197,3.355254e+10,1200000.0,849.415326,34.534287,1.104398,5.934351,41.877861,27.123427
2,Keller Williams Realty,8759,9.051790e+09,875000.0,568.826346,31.519808,1.008404,1.533303,40.113255,30.601781
3,Berkshire Hathaway HomeServices California Pro...,5813,8.747033e+09,950000.0,701.023125,41.650611,0.994020,0.980224,48.782983,29.749397
4,Intero Real Estate Services,4808,7.963182e+09,1377950.0,937.075078,24.655990,1.042946,3.382672,24.700499,26.131448
5,Coldwell Banker West,2876,7.121547e+09,825000.0,1812.412097,28.267038,3.094962,3.065508,38.274339,28.609527
6,First Team Real Estate,6249,7.045552e+09,960000.0,619.879691,29.540887,1.005647,97.301984,44.068971,30.629541
7,The Agency,2941,6.827176e+09,1680000.0,888.554040,35.456987,1.014694,1.003008,38.908285,26.760655
8,Real Broker,5176,5.453011e+09,849900.0,586.092624,29.412287,1.004521,194.197435,40.716577,30.086360
9,Pacific Sotheby's Int'l Realty,1791,5.370119e+09,1675000.0,1286.439276,39.026801,1.533072,1.499723,50.268565,28.932440


Top Buyer Offices


,BuyerOfficeName,closed_sales,total_sales_volume,median_close_price,average_price_per_sqft,average_days_on_market,average_price_ratio,average_close_to_original_list_ratio,average_listing_to_contract_days,average_contract_to_close_days
0,Compass,29602,5.397955e+10,1338000.0,869.656862,32.011756,1.054877,1.549973,37.628234,27.148950
1,Coldwell Banker Realty,16233,2.780711e+10,1199000.0,866.380245,34.704121,1.139759,1.375463,41.854228,27.451861
2,NaN,7152,9.434752e+09,1098000.0,855.277164,32.323826,1.031741,1.018607,28.250734,33.708852
3,Keller Williams Realty,6901,8.618636e+09,840000.0,776.772997,34.721635,1.303063,1.689836,43.178406,30.008261
4,Real Broker,6751,6.907365e+09,803000.0,586.408522,33.252259,1.017903,1.152376,43.608593,30.059852
5,Berkshire Hathaway HomeServices California Pro...,4514,6.792172e+09,983500.0,704.706150,39.500443,0.996910,1.194165,47.462716,29.803817
6,NONMEMBER MRML,9767,6.212295e+09,501990.0,359.020551,48.051705,0.995812,1.788871,57.295587,37.134637
7,First Team Real Estate,5667,6.187015e+09,900000.0,614.817638,31.024528,1.007978,173.162587,42.313504,30.473080
8,Intero Real Estate Services,3880,6.144827e+09,1335000.0,891.376298,25.055155,1.037324,1.026333,25.329209,26.346997
9,The Agency,2746,6.140777e+09,1550000.0,862.045827,38.506555,1.006591,0.997294,43.816588,29.129339


### 10. Export Week 6 Datasets

The final Week 6 datasets were exported after completing feature engineering,
school district mapping, and validation. 

The exported Sold and List datasets
retain all original analysis-ready records and include the newly engineered
market metrics and `DistrictName` field.

#### 10.1 Define Final Output Files

In [85]:
# Confirm the feature engineering output directory exists
FEATURE_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SOLD_ENRICHED_FILE = (
    FEATURE_OUTPUT_DIR
    / "sold_residential_features_enriched.csv"
)

LIST_ENRICHED_FILE = (
    FEATURE_OUTPUT_DIR
    / "list_residential_features_enriched.csv"
)

#### 10.2 Export Sold & List Final Datasets

In [86]:
# Export the final Week 6 feature-engineered datasets
sold_enriched.to_csv(
    SOLD_ENRICHED_FILE,
    index=False
)

list_enriched.to_csv(
    LIST_ENRICHED_FILE,
    index=False
)

#### 10.3 Validate Exported Datasets

In [87]:
# Confirm that the final datasets were successfully exported
print("Week 6 datasets exported successfully.")

print("\nSold dataset:")
print(SOLD_ENRICHED_FILE)
print("Shape:", sold_enriched.shape)

print("\nList dataset:")
print(LIST_ENRICHED_FILE)
print("Shape:", list_enriched.shape)

Week 6 datasets exported successfully.

Sold dataset:
cleaned_data/feature_engineering/sold_residential_features_enriched.csv
Shape: (447769, 99)

List dataset:
cleaned_data/feature_engineering/list_residential_features_enriched.csv
Shape: (615316, 83)


#### 10.4 Confirm Saved Column Structure

In [88]:
# Read only the headers to confirm the exported column structure
sold_export_check = pd.read_csv(
    SOLD_ENRICHED_FILE,
    nrows=0
)

list_export_check = pd.read_csv(
    LIST_ENRICHED_FILE,
    nrows=0
)

print(
    "Sold exported columns:",
    len(sold_export_check.columns)
)

print(
    "List exported columns:",
    len(list_export_check.columns)
)

print(
    "DistrictName saved in Sold:",
    "DistrictName" in sold_export_check.columns
)

print(
    "DistrictName saved in List:",
    "DistrictName" in list_export_check.columns
)

Sold exported columns: 99
List exported columns: 83
DistrictName saved in Sold: True
DistrictName saved in List: True
